# Real Data Sliding-Window Feature Dataset Builder

目标：读取指定原始数据路径下的 `npz` 和 `tdms` 文件，按 `0.02s` 时窗、`50%` 重叠切分，提取指定特征并导出特征数据集与日志。


In [25]:
from __future__ import annotations

from dataclasses import replace
from datetime import datetime, timedelta
from pathlib import Path
import logging
import os
import sys

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

os.environ.setdefault('FEA_CPT_USE_GPU', '1')

workspace = Path.cwd()
if not (workspace / 'src').exists():
    workspace = workspace.parent
if str(workspace / 'src') not in sys.path:
    sys.path.insert(0, str(workspace / 'src'))

from fea_cpt_gpu.base import FeatureRecord
from fea_cpt_gpu.features import compute_all_features
from fea_cpt_gpu.params import DEFAULT_FEATURE_PARAMS
from fea_cpt_gpu.signal_ops import build_context, butter_filter
from fea_cpt_gpu.gpu_backend import gpu_backend_info

try:
    from nptdms import TdmsFile
except ImportError:
    TdmsFile = None

print(f'workspace = {workspace}')
print(gpu_backend_info())
print('nptdms =', 'available' if TdmsFile is not None else 'missing')


workspace = e:\codes\ZZ-BK
[GPU] 使用 CUDA: NVIDIA GeForce RTX 4050 Laptop GPU
nptdms = available


In [26]:
# =========================
# Config (all user-editable)
# =========================
RAW_DATA_ROOT = Path(r'G:\\20260323_ZZ_pccp\\FIP\\24-900-1800\\fip-24下午')

WINDOW_DURATION_S = 0.02
WINDOW_OVERLAP = 0.50
assert 0.0 <= WINDOW_OVERLAP < 1.0

# 每处理多少个 npz/tdms 文件，新建一个 CSV 分片
NPZ_PER_CSV = 100

# 若 TDMS 文件的群组/通道名已知，可在这里指定；否则自动识别
TDMS_GROUP_NAME: str | None = None
TDMS_CHANNEL_NAME: str | None = None

# TDMS sample-rate fallback:
# 1) Try infer from filename text, e.g. 500K / 200k / 1M
# 2) If still unavailable and value is not None, use fixed fallback rate
TDMS_FALLBACK_SAMPLE_RATE_HZ: float | None = None

# 只保留这些特征列
SELECTED_FEATURES = [
    'b_1k_10k__SC_mean',
    'b_1k_10k__C_f',
    'b_1k_100k__SC_mean',
    'b_1k_10k__C_h',
    'b_40k_60k__SC_mean',
]

# 为得到上述列需要计算的频带
BANDS = [
    ('b_1k_100k', (1_000.0, 100_000.0)),
    ('b_1k_10k', (1_000.0, 10_000.0)),
    ('b_40k_60k', (40_000.0, 60_000.0)),
]

# 预处理带通，参照 2026-05-06-batch_feature_extract_12folders_gpu.ipynb
PREPROC_BAND = (1_000.0, 95_000.0)

OUTPUT_ROOT = workspace / 'outputs' / 'realdata_feature_dataset_20260519'
FEATURE_CSV_PREFIX = 'fip24afternoon_window_features'
LOG_CSV_PREFIX = 'fip24afternoon_window_log'
RUNTIME_LOG_NAME = 'fip24afternoon_runtime.log'
PROCESSED_NPZ_LIST_NAME = 'processed_npz_files.txt'

# 若只想快速试跑，可设置正整数；None 表示处理全部文件
MAX_FILES: int | None = None

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'RAW_DATA_ROOT = {RAW_DATA_ROOT}')
print(f'OUTPUT_ROOT   = {OUTPUT_ROOT}')
print(f'WINDOW_DURATION_S={WINDOW_DURATION_S}, WINDOW_OVERLAP={WINDOW_OVERLAP}')
print(f'NPZ_PER_CSV={NPZ_PER_CSV}')


RAW_DATA_ROOT = G:\20260323_ZZ_pccp\FIP\24-900-1800\fip-24下午
OUTPUT_ROOT   = e:\codes\ZZ-BK\outputs\realdata_feature_dataset_20260519
WINDOW_DURATION_S=0.02, WINDOW_OVERLAP=0.5
NPZ_PER_CSV=100


In [27]:
# =========================
# Helpers
# =========================

def build_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger('realdata_feature_dataset')
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')

    fh = logging.FileHandler(log_path, encoding='utf-8')
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger


def _safe_band(low: float, high: float, nyq: float) -> tuple[float, float]:
    low = max(1.0, min(low, nyq * 0.98))
    high = max(low + 1.0, min(high, nyq * 0.995))
    return (float(low), float(high))


def _scalar_text(value: object) -> str:
    if value is None:
        return ''
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='ignore').strip()
    if isinstance(value, np.generic):
        value = value.item()
    if hasattr(value, 'tolist') and not isinstance(value, str):
        try:
            value = value.tolist()
        except Exception:
            pass
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='ignore').strip()
    if isinstance(value, (list, tuple)) and len(value) == 1:
        return _scalar_text(value[0])
    return str(value).strip()


def _first_property(props: dict[str, object], names: tuple[str, ...]) -> object | None:
    normalized = {str(k).lower(): v for k, v in props.items()}
    for name in names:
        if name.lower() in normalized:
            return normalized[name.lower()]
    return None


def _coerce_float(value: object | None) -> float | None:
    if value is None:
        return None
    try:
        arr = np.asarray(value)
        if arr.shape == ():
            return float(arr.item())
        if arr.size == 1:
            return float(arr.reshape(()).item())
    except Exception:
        pass
    try:
        return float(value)
    except Exception:
        return None


def build_params_for_band(band: tuple[float, float], sample_rate: float):
    low, high = band
    nyq = sample_rate / 2.0
    low, high = _safe_band(low, high, nyq)
    span = max(high - low, 10.0)

    low_band = _safe_band(low, low + 0.30 * span, nyq)
    mid_band = _safe_band(low + 0.30 * span, low + 0.60 * span, nyq)
    high1_band = _safe_band(low + 0.50 * span, low + 0.80 * span, nyq)
    high2_band = _safe_band(low + 0.60 * span, high, nyq)
    harmonic_band = _safe_band(low + 0.50 * span, high, nyq)
    ridge_main = _safe_band(low, low + 0.65 * span, nyq)
    ridge_h2 = _safe_band(max(low * 2.0, low + 0.20 * span), min(high * 2.0, nyq * 0.995), nyq)

    return replace(
        DEFAULT_FEATURE_PARAMS,
        highpass_hz=1_000.0,
        main_band_hz=(low, high),
        low_band_hz=low_band,
        mid_band_hz=mid_band,
        high1_band_hz=high1_band,
        high2_band_hz=high2_band,
        harmonic_band_hz=harmonic_band,
        ridge_main_search_hz=ridge_main,
        ridge_h2_search_hz=ridge_h2,
        n_jobs=1,
    )


def preprocess_signal(raw_signal: np.ndarray, sample_rate: float) -> np.ndarray:
    demeaned = np.asarray(raw_signal, dtype=float) - float(np.mean(raw_signal))
    return butter_filter(demeaned, sample_rate=sample_rate, band_hz=PREPROC_BAND, order=4)


def parse_starttime(starttime_raw: str) -> datetime | None:
    if not starttime_raw:
        return None

    candidates = [
        '%Y%m%dT%H%M%S.%f',
        '%Y%m%dT%H%M%S',
        '%Y-%m-%d %H:%M:%S.%f',
        '%Y-%m-%d %H:%M:%S',
        '%Y-%m-%dT%H:%M:%S.%f',
        '%Y-%m-%dT%H:%M:%S',
    ]
    for fmt in candidates:
        try:
            return datetime.strptime(starttime_raw, fmt)
        except Exception:
            pass
    try:
        return datetime.fromisoformat(starttime_raw.replace('Z', '+00:00'))
    except Exception:
        return None


def iter_windows(signal_values: np.ndarray, sample_rate: float, window_duration_s: float, overlap: float):
    n = len(signal_values)
    window_samples = int(round(window_duration_s * sample_rate))
    if window_samples <= 0:
        raise ValueError('window_samples must be positive')
    if n < window_samples:
        return

    step_samples = int(round(window_samples * (1.0 - overlap)))
    step_samples = max(1, step_samples)

    idx = 0
    win_id = 0
    while idx + window_samples <= n:
        yield win_id, idx, idx + window_samples, window_samples, step_samples
        win_id += 1
        idx += step_samples


def _load_npz_source(path: Path) -> dict[str, object]:
    with np.load(path, allow_pickle=True) as data:
        signal_values = np.asarray(data['phase_data'], dtype=float)
        sample_rate = float(np.asarray(data['sample_rate']).item())
        starttime_raw = _scalar_text(data.get('starttime', '')) if 'starttime' in data else ''
        arrival_time_raw = _scalar_text(data.get('arrival_time', '')) if 'arrival_time' in data else ''
        sample_type = _scalar_text(data.get('type', path.parent.name)) if 'type' in data else path.parent.name

    return {
        'source_format': 'npz',
        'signal_values': signal_values,
        'sample_rate': sample_rate,
        'starttime_raw': starttime_raw,
        'arrival_time_raw': arrival_time_raw,
        'sample_type': sample_type,
        'source_group_name': '',
        'source_channel_name': '',
        'source_detail': '',
    }


def _select_tdms_channel(tdms_file):
    if TDMS_GROUP_NAME and TDMS_CHANNEL_NAME:
        for group in tdms_file.groups():
            if str(group.name).lower() == TDMS_GROUP_NAME.lower():
                for channel in group.channels():
                    if str(channel.name).lower() == TDMS_CHANNEL_NAME.lower():
                        return group, channel
        raise ValueError(f'Cannot find TDMS group/channel: {TDMS_GROUP_NAME}/{TDMS_CHANNEL_NAME}')

    if TDMS_CHANNEL_NAME:
        for group in tdms_file.groups():
            for channel in group.channels():
                if str(channel.name).lower() == TDMS_CHANNEL_NAME.lower():
                    return group, channel

    preferred_names = {'phase_data', 'signal', 'data', 'values', 'channel0', 'ch0'}
    for group in tdms_file.groups():
        for channel in group.channels():
            if str(channel.name).lower() in preferred_names:
                return group, channel

    best = None
    best_len = -1
    for group in tdms_file.groups():
        for channel in group.channels():
            try:
                values = np.asarray(channel[:])
                if values.size == 0:
                    continue
                if not np.issubdtype(values.dtype, np.number):
                    values = values.astype(float)
            except Exception:
                continue
            if values.size > best_len:
                best = (group, channel)
                best_len = values.size

    if best is None:
        raise ValueError('No usable numeric channel found in TDMS file')
    return best


def _infer_sample_rate_from_filename(path: Path) -> float | None:
    import re
    text = path.stem
    m = re.search(r'(?<!\d)(\d+(?:\.\d+)?)\s*([kKmM])(?![a-zA-Z])', text)
    if not m:
        return None
    value = float(m.group(1))
    unit = m.group(2).lower()
    if unit == 'k':
        return value * 1_000.0
    if unit == 'm':
        return value * 1_000_000.0
    return None
def _load_tdms_source(path: Path) -> dict[str, object]:
    if TdmsFile is None:
        raise ImportError('nptdms is required for .tdms files. Install with: pip install nptdms')

    tdms_file = TdmsFile.read(path)
    group, channel = _select_tdms_channel(tdms_file)

    signal_values = np.asarray(channel[:], dtype=float)
    combined_props: dict[str, object] = {}
    combined_props.update(getattr(tdms_file, 'properties', {}) or {})
    combined_props.update(getattr(group, 'properties', {}) or {})
    combined_props.update(getattr(channel, 'properties', {}) or {})

    sample_rate = _coerce_float(_first_property(combined_props, ('sample_rate', 'sample_rate_hz', 'sampling_rate', 'sampling_rate_hz')))
    if sample_rate is None:
        wf_increment = _coerce_float(_first_property(combined_props, ('wf_increment',)))
        if wf_increment and wf_increment > 0:
            sample_rate = 1.0 / wf_increment

    if sample_rate is None or sample_rate <= 0:
        sample_rate = _infer_sample_rate_from_filename(path)

    if (sample_rate is None or sample_rate <= 0) and TDMS_FALLBACK_SAMPLE_RATE_HZ is not None:
        sample_rate = float(TDMS_FALLBACK_SAMPLE_RATE_HZ)

    if sample_rate is None or sample_rate <= 0:
        raise ValueError(
            f'Cannot infer sample rate from TDMS file: {path}. ' 
            'Provide TDMS_FALLBACK_SAMPLE_RATE_HZ or include rate text like 500K in filename.'
        )

    starttime_raw = _scalar_text(_first_property(combined_props, ('starttime', 'start_time', 'wf_start_time', 'wf_starttime')))
    arrival_time_raw = _scalar_text(_first_property(combined_props, ('arrival_time', 'arrivaltime', 'arrival_time_text')))
    sample_type = _scalar_text(_first_property(combined_props, ('type', 'sample_type', 'sampletype'))) or path.parent.name

    return {
        'source_format': 'tdms',
        'signal_values': signal_values,
        'sample_rate': float(sample_rate),
        'starttime_raw': starttime_raw,
        'arrival_time_raw': arrival_time_raw,
        'sample_type': sample_type,
        'source_group_name': str(group.name),
        'source_channel_name': str(channel.name),
        'source_detail': f'{group.name}/{channel.name}',
    }


def load_source_file(path: Path) -> dict[str, object]:
    suffix = path.suffix.lower()
    if suffix == '.npz':
        return _load_npz_source(path)
    if suffix == '.tdms':
        return _load_tdms_source(path)
    raise ValueError(f'Unsupported file type: {path.suffix}')


def extract_window_features(window_signal: np.ndarray, sample_rate: float) -> dict[str, float]:
    filtered = preprocess_signal(window_signal, sample_rate)
    record = FeatureRecord(
        sample_id='window',
        sample_name='window',
        sample_type='raw',
        sample_type_code=0,
        path=Path('.'),
        signal=filtered,
        sample_rate=float(sample_rate),
        metadata={
            'source': 'sliding_window',
        },
    )

    out: dict[str, float] = {}
    for band_name, band in BANDS:
        params = build_params_for_band(band, sample_rate=record.sample_rate)
        context = build_context(record, params)
        result = compute_all_features(context)
        for k, v in result.features.items():
            out[f'{band_name}__{k}'] = float(v)
    return out





In [28]:
# =========================
# Main pipeline (append per file, split csv per 100 files)
# =========================
runtime_log_path = OUTPUT_ROOT / RUNTIME_LOG_NAME
processed_list_path = OUTPUT_ROOT / PROCESSED_NPZ_LIST_NAME
logger = build_logger(runtime_log_path)

source_files = sorted(
    p for p in RAW_DATA_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in {'.npz', '.tdms'}
)
if MAX_FILES is not None:
    source_files = source_files[:MAX_FILES]

if not source_files:
    raise FileNotFoundError(f'No npz/tdms files found under: {RAW_DATA_ROOT}')

processed_set: set[str] = set()
if processed_list_path.exists():
    processed_set = {ln.strip() for ln in processed_list_path.read_text(encoding='utf-8').splitlines() if ln.strip()}

logger.info('Found %d source files', len(source_files))
logger.info('Already processed: %d', len(processed_set))
logger.info('Window config: duration=%.6fs overlap=%.2f', WINDOW_DURATION_S, WINDOW_OVERLAP)
logger.info('Selected features: %s', ', '.join(SELECTED_FEATURES))
logger.info('NPZ_PER_CSV = %d', NPZ_PER_CSV)

window_total = 0
processed_now = 0


def chunk_paths(chunk_index: int) -> tuple[Path, Path]:
    suffix = f'part_{chunk_index:04d}.csv'
    feature_path = OUTPUT_ROOT / f'{FEATURE_CSV_PREFIX}_{suffix}'
    log_path = OUTPUT_ROOT / f'{LOG_CSV_PREFIX}_{suffix}'
    return feature_path, log_path


for file_idx, fp in enumerate(tqdm(source_files, desc='Files'), start=1):
    fp_str = str(fp)
    if fp_str in processed_set:
        continue

    chunk_index = (file_idx - 1) // NPZ_PER_CSV + 1
    feature_csv_path, log_csv_path = chunk_paths(chunk_index)

    source = load_source_file(fp)
    signal_values = np.asarray(source['signal_values'], dtype=float)
    sample_rate = float(source['sample_rate'])
    starttime_raw = str(source['starttime_raw'])
    arrival_time_raw = str(source['arrival_time_raw'])
    sample_type = str(source['sample_type'])
    source_format = str(source['source_format'])
    source_group_name = str(source.get('source_group_name', ''))
    source_channel_name = str(source.get('source_channel_name', ''))
    source_detail = str(source.get('source_detail', ''))

    start_dt = parse_starttime(starttime_raw)
    n_samples = len(signal_values)
    duration_s = n_samples / sample_rate if sample_rate > 0 else np.nan

    rows_features: list[dict[str, object]] = []
    rows_log: list[dict[str, object]] = []

    window_count = 0
    for win_id, i0, i1, win_len, step_len in iter_windows(signal_values, sample_rate, WINDOW_DURATION_S, WINDOW_OVERLAP):
        window_signal = signal_values[i0:i1]
        f_all = extract_window_features(window_signal, sample_rate)

        base_info = {
            'source_file_name': fp.name,
            'source_file_path': fp_str,
            'source_format': source_format,
            'source_group_name': source_group_name,
            'source_channel_name': source_channel_name,
            'source_detail': source_detail,
            'window_id': int(win_id),
            'window_start_index': int(i0),
            'window_end_index': int(i1),
            'window_length_samples': int(win_len),
            'window_step_samples': int(step_len),
            'window_duration_s': float(win_len / sample_rate),
            'window_start_offset_s': float(i0 / sample_rate),
            'sample_rate_hz': float(sample_rate),
            'source_n_samples': int(n_samples),
            'source_duration_s': float(duration_s),
            'starttime_raw': starttime_raw,
            'arrival_time_raw': arrival_time_raw,
            'sample_type': sample_type,
            'csv_chunk_index': int(chunk_index),
        }

        if start_dt is not None:
            base_info['window_start_datetime'] = (start_dt + timedelta(seconds=float(i0 / sample_rate))).strftime('%Y-%m-%d %H:%M:%S.%f')
        else:
            base_info['window_start_datetime'] = ''

        feature_row = dict(base_info)
        for feat_name in SELECTED_FEATURES:
            feature_row[feat_name] = float(f_all.get(feat_name, np.nan))
        rows_features.append(feature_row)

        log_row = dict(base_info)
        log_row['missing_selected_features'] = ','.join([f for f in SELECTED_FEATURES if f not in f_all])
        rows_log.append(log_row)
        window_count += 1

    df_features = pd.DataFrame(rows_features)
    df_log = pd.DataFrame(rows_log)

    feature_header = (not feature_csv_path.exists()) or (feature_csv_path.stat().st_size == 0)
    log_header = (not log_csv_path.exists()) or (log_csv_path.stat().st_size == 0)
    df_features.to_csv(feature_csv_path, mode='a', header=feature_header, index=False, encoding='utf-8-sig')
    df_log.to_csv(log_csv_path, mode='a', header=log_header, index=False, encoding='utf-8-sig')

    with processed_list_path.open('a', encoding='utf-8') as f:
        f.write(fp_str + '\n')
    processed_set.add(fp_str)
    processed_set.add(fp_str)

    window_total += len(df_features)
    processed_now += 1
    logger.info('Processed file=%s, format=%s, sample_rate=%.1fHz, n_samples=%d, windows=%d, chunk=%d', fp.name, source_format, sample_rate, n_samples, window_count, chunk_index)

logger.info('Run finished. Newly processed files=%d, total windows in this run=%d', processed_now, window_total)
logger.info('Processed list: %s', processed_list_path)
print('Done')
print(f'newly_processed_files = {processed_now}')
print(f'total_windows_this_run = {window_total}')
print(f'processed_list_path = {processed_list_path}')


[2026-05-19 13:14:08,200] INFO: Found 1516 source files
[2026-05-19 13:14:08,201] INFO: Already processed: 0
[2026-05-19 13:14:08,201] INFO: Window config: duration=0.020000s overlap=0.50
[2026-05-19 13:14:08,202] INFO: Selected features: b_1k_10k__SC_mean, b_1k_10k__C_f, b_1k_100k__SC_mean, b_1k_10k__C_h, b_40k_60k__SC_mean
[2026-05-19 13:14:08,202] INFO: NPZ_PER_CSV = 100


Files:   0%|          | 0/1516 [00:00<?, ?it/s]

[2026-05-19 13:19:10,009] INFO: Processed file=0000039-500K-20260324T134709.803.tdms, format=tdms, sample_rate=500000.0Hz, n_samples=5000000, windows=999, chunk=1


Files:   0%|          | 1/1516 [05:01<127:00:34, 301.80s/it]

[2026-05-19 13:24:17,966] INFO: Processed file=0000040-500K-20260324T134719.802.tdms, format=tdms, sample_rate=500000.0Hz, n_samples=5000000, windows=999, chunk=1


Files:   0%|          | 2/1516 [13:16<167:34:19, 398.45s/it]


KeyboardInterrupt: 

In [ ]:
# Quick check
feature_chunks = sorted(OUTPUT_ROOT.glob(f'{FEATURE_CSV_PREFIX}_part_*.csv'))
log_chunks = sorted(OUTPUT_ROOT.glob(f'{LOG_CSV_PREFIX}_part_*.csv'))
print('feature chunk count =', len(feature_chunks))
print('log chunk count =', len(log_chunks))
if feature_chunks:
    print('last feature chunk =', feature_chunks[-1])
if log_chunks:
    print('last log chunk =', log_chunks[-1])
